## Should I use [scvi-tools](https://scvi-tools.org/)? Or only 'traditional' Scanpy methods?

For your *current* task — two same-chemistry samples where Harmony already gave you clean mixing — scvi-tools is overkill. You don't need it to integrate this. But it's worth understanding what it buys you, because as your PDAC work scales up it becomes genuinely useful.

**When scanpy + a few external methods is enough (your situation now)**

- Few batches, same technology, moderate cell counts.
- Harmony/Scanorama already give good mixing (you confirmed this visually).
- You want speed, simplicity, and CPU-only reproducibility.

Harmony ran in seconds and worked. Adding a VAE that needs training, a GPU to be practical, and more hyperparameters to get the same embedding is effort without payoff here. Don't switch just because scvi-tools is fashionable.

**When scvi-tools earns its place**

1. **Atlas-scale integration with many batches / strong batch structure.** scVI's amortized VAE scales to millions of cells and handles complex, non-linear batch effects better than Harmony's linear PC adjustment. This is where the benchmarks favor it.

2. **You have (partial) cell-type labels → scANVI.** Semi-supervised integration that preserves biology while transferring labels to unlabeled cells. If you build a PDAC reference and want to annotate new samples against it, this is the tool.

3. **Batch-aware differential expression in one model.** This is the big one for you. Earlier we discussed the embedding-vs-DE split — scVI's `model.differential_expression()` does DE *in the same generative model*, correcting for batch natively, without you running pseudobulk separately. For cross-sample contrasts in tumors it's a clean, principled alternative to DESeq2-on-pseudobulk.

4. **Reference mapping (scArches).** Map new query samples onto a trained reference without retraining — powerful once you have a PDAC atlas others will query.

5. **Multi-modal / specialized models.** totalVI (CITE-seq), PeakVI (ATAC), MultiVI, DestVI and Stereoscope (spatial deconvolution). Given your bulk RNA-seq deconvolution background, **DestVI/Stereoscope/Cell2location** are directly relevant if you move toward spatial or single-cell-informed deconvolution.

**A practical hybrid — which is what most people actually do**

You don't choose one *or* the other. Scanpy is the backbone (I/O, QC, HVG, plotting, clustering, DE); scvi-tools is a specialized engine you call for integration/DE when the problem warrants it, writing its latent space back into the same AnnData:

```python
import scvi
scvi.model.SCVI.setup_anndata(adata, layer="counts", batch_key="sample")
model = scvi.model.SCVI(adata)
model.train()
adata.obsm["X_scVI"] = model.get_latent_representation()

# then back to scanpy on the scVI space
sc.pp.neighbors(adata, use_rep="X_scVI")
sc.tl.umap(adata)
sc.tl.leiden(adata)
```

Note it consumes `layer="counts"` (raw, not log-norm) — scVI models the count distribution directly, which is one of its strengths and one reason keeping your counts layer matters.

**My recommendation for you:**

- **Right now:** stay with scanpy + Harmony. It works, it's clean, don't add complexity.
- **Add scvi-tools when** you hit any of: (a) many batches / a real atlas, (b) you want reference-based annotation via scANVI, (c) you want model-based batch-aware DE, or (d) you move toward spatial/deconvolution (DestVI/cell2location — most aligned with your bulk-deconvolution expertise).

One honest caveat: scVI results depend on training (epochs, latent dim, seed) and aren't deterministic the way Harmony is, so for a two-sample analysis you'd also lose some reproducibility for no benefit. Save it for where its strengths matter.

If you tell me where this PDAC single-cell work is heading — building a reference, spatial follow-up, or cross-condition DE — I can point you to the specific scvi-tools model rather than the whole menu.